# **Lab:** Security in Computer Networks

**Student name:** Mateo Jiménez

### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures

In [1]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [3]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

Keys generated successfully


In [4]:
# Export public key to PEM
pem_pub = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

pem_pub

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAwl3fKpsu5nENw7edYf1U\nVwwkqArf6IVu8+GKGeTEnW0bS+EQnGpUR390ogZULmAzhZDqIOFRPLU/dzv31hH1\nF1bfdwy7KQXObyRMlV9yfghYbU5PMgfi6I1JR5WPZ0z1ioH4TVRQEV1WaV15DoY7\nqDhyuI6RfYWHWI7Ehwct0NkX4XqWmezkZHWGVic0Kt1On9TZevLZAi53/1ayLcYa\nS+lRicsrqOoXKg07FOylSiS7mkRCRqGMb8tTYdkh7M1rfXjjuZRU6ZhmWTqpsDvv\ni9ypGhj0JOD6zPGCda/qc+mlx4a7+ch5bFOyiASsHeVMsup3HnlxVr/2gXjy+Pos\ngQIDAQAB\n-----END PUBLIC KEY-----\n'

In [5]:
# Convert received PEM to an object
received_public_key = serialization.load_pem_public_key(pem_pub.encode())

received_public_key

In [6]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "Hello secure world".encode()

# Encrypt using PUBLIC key
encrypted = received_public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

Encrypted (Python bytes object): b'\x8f\xb5\x9b\xb4\xd3\x1a\xc7\x08-\xcb\xb1"C`X\xba\xf6\x8b\xc0\xe4\xf6Y\x81\x8a\xf1\xd6h_\xc0\xc1g\xf0\xc5\x1b\xb7\x81\xea\x9eP\xae\x13\x83\xff\xe3\x0e\xf2\xca6\xe0m\xe4\xfb\xf1\x01>YA6\xe1\xfc\x7f\xfen\xb4\x15i2pM\xa1\x9b\x11\xf6\xf1\xedy\x1c\x8f084~S*RM\xf6E\xbcA\xb6[\x19~\xba\xad\xe9\x9c\x93B\xe0l@\xf8g{\x92\x8f\xa6\xe6f4\x95Rh\xcb\xe3\xbd\xa13ge[\x9a\xef\\NG\\\xaf\xbe\xe1\x19\xcb\xe2\xf0Y\xd2\x85T\xb9(\xcehu\xd5+\xca\x08T]\xbaD2q\xa5\xf9\xbd6&\xcaQ[\xdd\xeeT\x89\x18H\x93u\xa8\x91\xfd\x1c\xfc\xea\xf4:q\xc2\xf8\x1b\xbd\xed\xf7\x87\xc3A\x97D\x80\x1c[\x80*BWvy\x8c\x12\x9f.t\xc8\n\xf7\x83)t\xd7\x89\xe06\x1d6hLb\xc7\xd4A\xec\x17o&,\xbe\x92\x9dU\x16\x7f&\x8f\xe8>o\x81\x17Zo\xbd\xb8>\x9a\xafe4\xfbU\x06\x98\xdf\x9c'
Encrypted (hex): 8fb59bb4d31ac7082dcbb122436058baf68bc0e4f659818af1d6685fc0c167f0c51bb781ea9e50ae1383ffe30ef2ca36e06de4fbf1013e594136e1fc7ffe6eb4156932704da19b11f6f1ed791c8f3038347e532a524df645bc41b65b197ebaade99c9342e06c40f8677b928fa6e666349552

In [7]:
# Decrypt using PRIVATE key
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

Decrypted: Hello secure world


When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [8]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")

Signature created


In [9]:
# Verify signature
try:
    public_key.verify(
        signature,  # signature to verify
        message,    # original message
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

Signature VALID


How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [10]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [11]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

Session key created


In [12]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [13]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [14]:
# Secure communication
secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

Encrypted message: b'gAAAAABq-0pDNXjPqmrpwQhA8KsZwR6jQeFfdxGxZXll5R8pPEeqSTyvV1MgCD4Xm0ZLTBVmlRGgQAgmsVR1dmqpyCwlz7zxFlxNBfZYMtlW4BpEQoSLJb8='
Decrypted message: Hello TLS secure world


## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)

### **Group Dynamics (Balanced Work)**

Each student must complete ALL roles:

1. **Sender**
2. **Receiver**

You will interact with at least **one other student**.

### **Task 1:** Secure Message Sending

Each student must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another student:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each student must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| **Criteria** | **Score (10pts)** |
|--------|------|
| RSA encryption implemented | 1 pts |
| Digital signature implemented | 1 pts |
| Correctly decrypted & verified message | 4 pts |
| Explanation reflection questions | 4 pts |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security?  
2. What is the role of digital signatures?  
3. Why does TLS use a hybrid approach?  

---
# ✅ Solution — Secure Communication Challenge

Below I implement the assignment as reusable **Sender** and **Receiver** functions.

* The message is **encrypted with the receiver's public key** (RSA-OAEP + SHA-256) → *confidentiality*.
* The message is **signed with the sender's private key** (RSA-PSS + SHA-256) → *authentication + integrity*.
* Everything that travels over the network (ciphertext, signature, public key) is converted to **Base64 / PEM text** so it can be pasted into a chat, e-mail or file.

To make the notebook runnable on its own, my classmate is simulated in the notebook (`partner_*` keys).
When doing the exchange live, paste the real values from your classmate in the cells marked **📥 PASTE HERE**.

In [20]:
import base64, json

# ---------- Key helpers ----------
def generate_keys():
    """Generate an RSA-2048 key pair (private, public)."""
    priv = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    return priv, priv.public_key()

def public_key_to_pem(pub):
    """Public key -> PEM text (this is the ONLY key we share)."""
    return pub.public_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PublicFormat.SubjectPublicKeyInfo
    ).decode()

def pem_to_public_key(pem_text):
    """PEM text received from someone else -> public key object."""
    return serialization.load_pem_public_key(pem_text.encode())

OAEP = padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()),
                    algorithm=hashes.SHA256(), label=None)
PSS  = padding.PSS(mgf=padding.MGF1(hashes.SHA256()),
                   salt_length=padding.PSS.MAX_LENGTH)

# ---------- SENDER ----------
def send_secure_message(text, receiver_public_key, sender_private_key):
    """Encrypt with receiver's PUBLIC key and sign with sender's PRIVATE key."""
    data = text.encode()
    encrypted = receiver_public_key.encrypt(data, OAEP)          # confidentiality
    signature = sender_private_key.sign(data, PSS, hashes.SHA256())  # authenticity + integrity
    return {
        "encrypted_message": base64.b64encode(encrypted).decode(),
        "signature":         base64.b64encode(signature).decode(),
        "sender_public_key": public_key_to_pem(sender_private_key.public_key()),
    }

# ---------- RECEIVER ----------
def receive_secure_message(package, receiver_private_key):
    """Decrypt with my PRIVATE key and verify with the sender's PUBLIC key."""
    encrypted = base64.b64decode(package["encrypted_message"])
    signature = base64.b64decode(package["signature"])
    sender_pub = pem_to_public_key(package["sender_public_key"])

    try:
        plaintext = receiver_private_key.decrypt(encrypted, OAEP)
    except Exception:
        print("❌ REJECTED: could not decrypt (message was not encrypted for me or was modified)")
        return None

    try:
        sender_pub.verify(signature, plaintext, PSS, hashes.SHA256())
    except Exception:
        print("❌ REJECTED: signature INVALID (wrong sender or message altered)")
        return None

    print("✔ ACCEPTED: signature VALID")
    print("Message:", plaintext.decode())
    return plaintext.decode()

print("Helper functions ready")

Helper functions ready


## Task 1 — Secure Message Sending (role: **Sender**)

### Step 1 & 2 — Generate my key pair and share ONLY my public key

In [21]:
my_private_key, my_public_key = generate_keys()

my_public_pem = public_key_to_pem(my_public_key)
print("📤 My PUBLIC key (send this to my classmate):\n")
print(my_public_pem)
# my_private_key is NEVER printed or shared

📤 My PUBLIC key (send this to my classmate):

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAnSZJvZgA4Al4N5Ddpl0g
ZpvqvkugjCKa8GM8ijJFwEYuAayvxLScM55Mn/2a2/fFm9jGBSSywHt8r6H/m+gv
eTiaHEZSMRi/y7grHeBC+LpIS69bdbzjm/psMp/ZB7TW8r5tZygvCs5RHZ0lvsX+
NE2MpDLfC2qmsCw463l8Uec/ICmHBgEGGFsKhnlg02DOo1BUKrogCcCR7UwW/iSd
JDdWg8eJNkr8gkNmb8JRgz/yoLuJ/J9Emr7YT+WmTqWakBtcfNtcKA3oQUJO1JQx
QugSxDM2TYLMgF6lMxSHyydmUgt+2MOpzgkHVXCV0lhj4hZnppGnajIdKArqh44Y
jQIDAQAB
-----END PUBLIC KEY-----



### Receiving my classmate's public key
**📥 PASTE HERE** the PEM public key your classmate sends you. If it is left empty, a simulated classmate is created so the notebook still runs.

In [22]:
partner_public_pem = """-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAvmYu8XnhDERfegFL4/Be
bxzbXjsMXDVuF0s/B74Aw31H/xh+Fzmuzs8uGX1oNX4fhKoOKqTGwZvnYwVDet9v
tHxGcxjn2tAXiuoqhIdhf3WB4UBeoSHVBMQYJEpxoqpwWnLuoMQpubuZr2jJvN0V
rP8Bz6L5y5vQn9zqq/lRSYkY0Yi4Bw9iU6VlSx1rKHitwNmMownxX2JU0SMBVYqx
2CDr4sVWwupXPqVdH0dXl7ImnB8qbKdzpVCzBITDIuOldPUWNmBg/WhYJNISx9v1
uAD5gBt9r24hxH4RC77T+EjyKjpEDIpT8R25GrruStqJP5masc6p8IPA28htzCVQ
XQIDAQAB
-----END PUBLIC KEY-----"""   # 📥 paste classmate's public key (-----BEGIN PUBLIC KEY----- ...)

if partner_public_pem.strip():
    partner_public_key = pem_to_public_key(partner_public_pem)
    partner_private_key = None          # we never have the classmate's private key
    print("Using classmate's real public key")
else:
    # Simulated classmate (only for testing the notebook alone)
    partner_private_key, partner_public_key = generate_keys()
    partner_public_pem = public_key_to_pem(partner_public_key)
    print("Simulated classmate created")

Using classmate's real public key


### Steps 3, 4 & 5 — Create the message, encrypt it with the receiver's public key and sign it with my private key

In [23]:
my_message = "Hi! This is Mateo. Meet at the networks lab at 10:00."

package_to_send = send_secure_message(my_message, partner_public_key, my_private_key)

print("👉 Send this to my classmate:\n")
print(json.dumps(package_to_send, indent=2))

# Also save it to a file so it can be exchanged easily
with open("package_from_mateo.json", "w") as f:
    json.dump(package_to_send, f, indent=2)
print("\nSaved to package_from_mateo.json")

👉 Send this to my classmate:

{
  "encrypted_message": "TxYk6Kw0XQ4MHcCR5/IcD8KEQz6Pk2vZ0i6bE8yadARQmBKT0LyMQARAvPiqeziB6ErK00HyaWP8FLS4uigKyvWe8DkzTzz4aHDLrDoRdyGZYVZ3eHvNpS61ZOI9CeoueUU4V0SKcr2nx8uNDO96bDjxrf7VcwHsRZekolGzkNPpF0mTnSAEJWTVvbG95gg36G3FIS321eHQokoGnqviloXrGIZ2j4qlAwsBMovyjuMoIdx9ZvcZr7Bfa3U5wGhtn5OUsNaVRt2y9ASUlqa1vLMtc90xgbQyrS+j7VhYlXSL0dXS9wkFMNi6QklSLp7L7ORrDl/pTFcGgmzwdGuntw==",
  "signature": "AbHq2EpzMNAqIJbwVy4IzPq+KIbcSYuik8B49PQfaqXSLaYfzbJewD97mz90dFFa7ZWW9LjQl/Z8Xy69XCKvvUnfw8iqLvanTgoM6t1B08WudecOJpIbT+8z9ADYZ55XgHZc5pX6aThGxlWjrGRMwSXDV7lTZrIo9fWth5mf4v40Pk5Hp/WbNeEWfLsWTu8Z0grVpSvZqqc4r17takdwtr2H2R+B9JXBXww2xZm9HxDVFY37xg61Fm9FP163lQmq+qL18N/qRnq/D7aTh0GQgKsvAjpQ0EwPOS2nSY9t5tRKQp1K9FVOwac4DtGRa4ACzCZtkDPJ7vhxQDFdVB9PYw==",
  "sender_public_key": "-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAnSZJvZgA4Al4N5Ddpl0g\nZpvqvkugjCKa8GM8ijJFwEYuAayvxLScM55Mn/2a2/fFm9jGBSSywHt8r6H/m+gv\neTiaHEZSMRi/y7grHeBC+LpIS69bdbzjm/psMp/ZB7TW8r5tZy

## Task 2 — Secure Message Receiving (role: **Receiver**)

**📥 PASTE HERE** the package your classmate sends you (encrypted message, signature and their public key — all as text).
If left empty, the simulated classmate sends me a message.

In [24]:
received_package = {
  "sender": "Camilo Vasquez",
  "sender_public_key": "-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAvmYu8XnhDERfegFL4/Be\nbxzbXjsMXDVuF0s/B74Aw31H/xh+Fzmuzs8uGX1oNX4fhKoOKqTGwZvnYwVDet9v\ntHxGcxjn2tAXiuoqhIdhf3WB4UBeoSHVBMQYJEpxoqpwWnLuoMQpubuZr2jJvN0V\nrP8Bz6L5y5vQn9zqq/lRSYkY0Yi4Bw9iU6VlSx1rKHitwNmMownxX2JU0SMBVYqx\n2CDr4sVWwupXPqVdH0dXl7ImnB8qbKdzpVCzBITDIuOldPUWNmBg/WhYJNISx9v1\nuAD5gBt9r24hxH4RC77T+EjyKjpEDIpT8R25GrruStqJP5masc6p8IPA28htzCVQ\nXQIDAQAB\n-----END PUBLIC KEY-----\n",
  "encrypted_message": "Gkuj8wNFeP7NvuntRJ0C2iYxcEJh9Nfex7W6nywlkJY3jFt3XLefbEBYnRj2owvUTfAI1q87718X8DLoEEyCmIfvDw6CFUSmLjZs6OnIZswwEHk2deSqZwBKPVQpGcpPIPnIQlRdTamAsffXaAbS3IQbEPL1OlDXXd1H2Absoeo8/enxOjeuxFOKNSwO9M1d88sIHqJaGpR02o/cMruBG2c6cXZhNfit9R71XbpSfrwdNv90efVQmM6Z/oaLu3vYfOv5dhoDpurUFtP45GSvPlqMctkncl5ZkxdOZ/rYoCiqTuPeeUD8AvPYWX0GV/iSpkaclX+C0XrSjRhlWzvNVA==",
  "signature": "muTdFBsMX/+SA2DOvv8zN6R+svcVnS1lX89dpv4SE1YPcbUKv9CIpdMWPu7bI4V7U5TNMWfMc54zgq2Q5e33oNaNdpwibh1tYJnZ8K17LOYQrSYRRuxlGfbWqhlsEvoGN/Zmerav0MqOiuacSxmfJiQDVDAvLQ6D2Z7YV85jNwGvUxOzpEmBk8BQ2W+Gu0ARAmqE/xUT1l9dk2jVMjroKO/3K8GOWa/4NaNJCBf+FxlCuNIhKAR/BqSpPGlIhPwT0+121oPDU36x8/yi5yRtIUmAjlMw93jY23EM7hT5vTWKE7Ibz398KgLlj3H6yuXoQuzkdu+4FLdwpvWJtjmnoQ=="
}

if not received_package["encrypted_message"]:
    # Simulated classmate: encrypts with MY public key and signs with THEIR private key
    received_package = send_secure_message(
        "Hello Mateo, message received. See you at 10:00!",
        my_public_key, partner_private_key)
    print("Simulated package received:\n")
    print(json.dumps(received_package, indent=2)[:600], "...")

### Steps 2 & 3 — Decrypt with my private key and verify the signature with the sender's public key

In [25]:
accepted = receive_secure_message(received_package, my_private_key)

✔ ACCEPTED: signature VALID
Message: Hi! This is Camilo. Meet at the networks lab at 10:00.


### Checking that the receiver really rejects bad messages
To prove that verification works, I test three manipulated packages:

In [27]:
print("Test A — signature replaced by an attacker's signature")
attacker_priv, _ = generate_keys()
bad = dict(received_package)
bad["signature"] = base64.b64encode(
    attacker_priv.sign(b"Hello Mateo, message received. See you at 10:00!", PSS, hashes.SHA256())).decode()
receive_secure_message(bad, my_private_key)

print("\nTest B — attacker re-encrypts a NEW message with my public key but keeps the original signature")
bad = dict(received_package)
bad["encrypted_message"] = base64.b64encode(
    my_public_key.encrypt(b"Send me your password", OAEP)).decode()
receive_secure_message(bad, my_private_key)

print("\nTest C — one byte of the ciphertext flipped in transit")
raw = bytearray(base64.b64decode(received_package["encrypted_message"]))
raw[10] ^= 0xFF
bad = dict(received_package)
bad["encrypted_message"] = base64.b64encode(bytes(raw)).decode()
receive_secure_message(bad, my_private_key)

Test A — signature replaced by an attacker's signature
❌ REJECTED: signature INVALID (wrong sender or message altered)

Test B — attacker re-encrypts a NEW message with my public key but keeps the original signature
❌ REJECTED: signature INVALID (wrong sender or message altered)

Test C — one byte of the ciphertext flipped in transit
❌ REJECTED: could not decrypt (message was not encrypted for me or was modified)


## Reflection Questions

### 1. Why is encryption alone NOT enough for security?
Encryption only gives **confidentiality**: it hides the content from third parties. It does **not** tell the receiver *who* sent the message or whether it was changed. Anyone can encrypt a message with my public key (it is public), so an attacker could send me a fake message pretending to be my classmate, or replace the ciphertext in transit with another one. Encryption also does not prove that the public key I used really belongs to the intended person (Man-in-the-Middle). Real security needs confidentiality **plus** authentication, integrity and trusted keys (certificates).

### 2. What is the role of digital signatures?
A digital signature is created with the sender's **private key** and verified with their **public key**. It provides:
- **Authentication** – only the owner of the private key could have produced it, so it proves who sent the message.
- **Integrity** – the signature is computed over a hash (SHA-256) of the message; if a single bit changes, verification fails (Tests A, B and C above).
- **Non-repudiation** – the sender cannot later deny having signed it.

Signatures do *not* hide the message; that is why in the assignment we combine them with encryption.

### 3. Why does TLS use a hybrid approach?
Asymmetric crypto (RSA) solves the **key-distribution problem** — two parties that never met can agree on a secret over an insecure network — but it is **slow** and can only encrypt very small data (≈190 bytes with RSA-2048 + OAEP). Symmetric crypto (AES / Fernet) is **very fast** and can encrypt any amount of data, but both parties need the same secret key. TLS combines both: public-key crypto (plus certificates) to authenticate the server and securely establish a **session key**, and then fast symmetric encryption with that session key for all the actual traffic. (Modern TLS 1.3 uses ephemeral Diffie-Hellman instead of RSA key transport, but the idea is the same.)